In [3]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow import keras

# Créer un dataset
x, y = make_classification(n_samples=1000,
                           n_features=20,
                           n_informative=10,
                           n_redundant=5,
                           n_classes=2,
                           random_state=0)

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

# Définir le modèle à créer (attention à la signature pour scikeras !)
def create_model(numberOfHiddenUnits=16, optimizer='adam'):
    model = keras.Sequential([
        keras.layers.Dense(units=numberOfHiddenUnits, activation='relu', input_shape=(20,)),
        keras.layers.Dense(units=1, activation='sigmoid')
    ])
    model.compile(optimizer=optimizer,
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

# Wrap avec scikeras
model = KerasClassifier(model=create_model, epochs=10, batch_size=32, verbose=0)

# Grille des hyperparamètres à tester
params = {
    'model__numberOfHiddenUnits': [16, 32, 64],
    'model__optimizer': ['adam', 'rmsprop']
}

# Grid Search
grid = GridSearchCV(estimator=model, param_grid=params, cv=3)
grid_result = grid.fit(x_train, y_train)

print("Meilleur score : %.4f avec %s" % (grid_result.best_score_, grid_result.best_params_))

rs = RandomizedSearchCV(estimator=model,param_distributions=params, cv=3,n_iter=5)
randSearchCv = rs.fit(x_train,y_train)

print("Meilleur score : %.4f avec %s" % (randSearchCv.best_score_, randSearchCv.best_params_))
# Afficher les meilleurs résultats


Meilleur score : 0.8188 avec {'model__numberOfHiddenUnits': 64, 'model__optimizer': 'rmsprop'}
Meilleur score : 0.8025 avec {'model__optimizer': 'adam', 'model__numberOfHiddenUnits': 64}
